In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)


In [0]:
dbutils.widgets.text('catalog','commerce_stage_dev')
dbutils.widgets.text('schema','silver')
dbutils.widgets.text("env", "dev")

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
env = dbutils.widgets.get("env")

In [0]:
customer_df = spark.table(f'commerce_raw_{env}.bronze.customers')

In [0]:
display(customer_df)

In [0]:
from pyspark.sql.functions import col
customer_filter_df = customer_df.filter(col("customer_id").isNotNull())

In [0]:
customer_clean_df = customer_filter_df.distinct()

In [0]:
customer_clean_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{schema}.customers_temp1")

In [0]:
%sql
select * from customers_temp1

In [0]:
spark.sql(f"""
CREATE TABLE if not EXISTS {catalog}.{schema}.customer_stage_dedup as
select *  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY ingestion_time DESC
           ) AS rn
    FROM customers_temp1
)
WHERE rn = 2
""");


In [0]:
spark.sql(f"""
ALTER TABLE commerce_stage_{env}.{schema}.customer_stage
ALTER COLUMN customer_id TYPE STRING
""")

In [0]:
spark.sql(f"""
          MERGE INTO commerce_stage_{env}.{schema}.customer_stage cs
          USING commerce_stage_{env}.{schema}.customer_stage_dedup cd
ON cs.customer_id = cd.customer_id

WHEN MATCHED THEN
UPDATE SET
    cs.customer_name = cd.customer_name,
    cs.email = cd.email,
    cs.city = cd.city,
    cs.state = cd.state,
    cs.registration_date = cd.signup_date,
    cs.updated_ts = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    customer_name,
    email,
    city,
    state,
    registration_date,
    created_ts,
    updated_ts
)
VALUES (
    cd.customer_id,
    cd.customer_name,
    cd.email,
    cd.city,
    cd.state,
    cd.signup_date,
    current_timestamp(),
    current_timestamp()
)
          """)